# ACUARIO-IA · Fase 4: monitoreo de temperatura

**Prototipo educativo con datos simulados.** Compara una temperatura ficticia con el rango ilustrativo de **24 °C a 26 °C**, genera alertas y representa una corrección simulada. No usa sensores ni controla equipos reales.

Incluye un acuario con peces animados, termómetro y grafo de estados que muestra visualmente el recorrido. La lógica usa reglas de umbral y una ruta simple; no es aprendizaje automático ni A*.

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

TEMP_MIN, TEMP_MAX = 24, 26

def diagnosticar(t):
    if t < TEMP_MIN: return 'ALERTA: temperatura baja', 'calentar'
    if t > TEMP_MAX: return 'ALERTA: temperatura alta', 'enfriar'
    return 'NORMAL: temperatura en rango', 'mantener'

def crear_ruta(t):
    ruta = [int(t)]
    while ruta[-1] > TEMP_MAX: ruta.append(ruta[-1] - 1)
    while ruta[-1] < TEMP_MIN: ruta.append(ruta[-1] + 1)
    return ruta

print('Configuración lista. Rango educativo:', TEMP_MIN, '°C a', TEMP_MAX, '°C')

## Simulador visual animado
Mueve el control y pulsa **Ejecutar simulación**. Los peces se mueven y el grafo ilumina la ruta de corrección. Cada paso simula un ajuste de 1 °C.

In [ ]:
temperatura_control = widgets.IntSlider(value=28, min=18, max=32, step=1, description='Temperatura:', continuous_update=False, style={'description_width':'initial'}, layout=widgets.Layout(width='420px'))
boton_simular = widgets.Button(description='▶ Ejecutar simulación', button_style='success', icon='play')
salida = widgets.Output()

def ejecutar_simulacion(_=None):
    with salida:
        clear_output(wait=True)
        inicial = int(temperatura_control.value)
        ruta = crear_ruta(inicial)
        estado, accion = diagnosticar(inicial)
        print(f'LECTURA INICIAL: {inicial} °C')
        print(f'RANGO ILUSTRATIVO: {TEMP_MIN}–{TEMP_MAX} °C')
        print('ESTADO:', estado)
        print('DECISIÓN:', 'mantener monitoreo' if accion == 'mantener' else f'alertar y simular acción de {accion}')
        print('RECORRIDO DEL GRAFO:', ' → '.join(f'{t} °C' for t in ruta))
        print('Demostración académica: no envía órdenes a equipos reales.')

        fig, (ac, gr) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios':[1,1.25]})
        fig.patch.set_facecolor('#071d33')
        for ax in (ac,gr):
            ax.set_facecolor('#071d33'); ax.tick_params(colors='white')
            for sp in ax.spines.values(): sp.set_color('#35536d')

        # Acuario dibujado: agua, plantas, peces y termómetro
        ac.set_xlim(0,10); ac.set_ylim(0,10); ac.set_xticks([]); ac.set_yticks([])
        ac.set_title('ACUARIO SIMULADO',color='white',fontsize=13)
        ac.fill_between([.5,8.1],[1,1],[8.5,8.5],color='#0b6388',alpha=.6)
        ac.plot([.5,.5,8.1,8.1],[1,8.5,8.5,1],color='#76d8ed',lw=2); ac.plot([.5,8.1],[1,1],color='#76d8ed',lw=2)
        for x in [1.2,2.1,7.2]: ac.plot([x,x-.2,x+.15],[1.2,2,2.7],color='#43c59e',lw=3)
        ac.scatter([1.5,3.1,5.5,7.1],[1.1]*4,s=35,color='#e5c07b')
        specs=[(2,5.8,'#ffb347',.42),(5,4,'#55d6d2',.35),(3.2,7,'#ff7b9c',.28)]
        bodies=[]; tails=[]; eyes=[]
        for x,y,c,size in specs:
            bodies.append(ac.scatter([x],[y],s=600*size/.4,color=c,edgecolors='white',zorder=4))
            tails.append(ac.scatter([x-.35],[y],s=150*size/.4,marker='>',color=c,zorder=3))
            eyes.append(ac.scatter([x+.12],[y+.06],s=10,color='#071d33',zorder=5))
        ac.add_patch(plt.Rectangle((8.65,2.1),.45,5.2,fill=False,edgecolor='white',lw=2))
        ac.add_patch(plt.Circle((8.875,1.95),.42,color='#f05d5e',ec='white',lw=1.5))
        mercury=ac.add_patch(plt.Rectangle((8.79,2.15),.17,4.5,color='#f05d5e'))
        temp_text=ac.text(5,9.1,'',ha='center',color='white',fontsize=12,fontweight='bold')
        state_text=ac.text(5,.35,'',ha='center',color='#8be9fd',fontsize=9)

        # Grafo lineal de temperaturas posibles y ruta activa
        estados=list(range(18,33)); gr.set_xlim(17,33); gr.set_ylim(-1.5,2.6); gr.set_yticks([]); gr.set_xticks(estados[::2])
        gr.set_xlabel('Temperatura del agua (°C)',color='white'); gr.set_title('GRAFO DE DECISIÓN',color='white',fontsize=13)
        gr.axvspan(TEMP_MIN,TEMP_MAX,color='#43c59e',alpha=.2,label='Rango aceptable')
        for t in estados[:-1]: gr.annotate('',xy=(t+.85,0),xytext=(t+.15,0),arrowprops={'arrowstyle':'->','color':'#29445d','lw':1})
        gr.scatter(estados,[0]*len(estados),s=70,color='#49647d',zorder=2)
        gr.scatter([TEMP_MIN,TEMP_MAX],[0,0],s=110,color='#43c59e',edgecolors='white',zorder=3)
        gr.text(25,1.6,'Comparar con el rango 24–26 °C',ha='center',color='white',fontsize=10)
        ruta_line,=gr.plot([],[],color='#36e0a1',lw=3,marker='o',markersize=7,zorder=5)
        nodo=gr.scatter([],[],s=180,color='#ffd166',edgecolors='white',zorder=6)
        etiqueta=gr.text(25,-.9,'',ha='center',color='white',fontsize=10)
        gr.legend(loc='upper right',facecolor='#102a43',labelcolor='white',fontsize=8)
        frames=max(24,len(ruta)*12)

        def actualizar(frame):
            paso=min(len(ruta)-1,frame//12); temp=ruta[paso]
            for i,(body,tail,eye,spec) in enumerate(zip(bodies,tails,eyes,specs)):
                x0,y,c,size=spec; x=.9+((x0+frame*.055*(1 if i%2==0 else -1)-.9)%6.8); yy=y+.12*math.sin(frame*.15+i)
                body.set_offsets([[x,yy]]); tail.set_offsets([[x-.35,yy]]); eye.set_offsets([[x+.12,yy+.06]])
            temp_text.set_text(f'Temperatura: {temp} °C')
            if TEMP_MIN <= temp <= TEMP_MAX: state_text.set_text('✓ EN RANGO · monitoreo'); state_text.set_color('#72f1b8')
            else: state_text.set_text('⚠ ALERTA · corrección simulada'); state_text.set_color('#ff7272')
            mercury.set_height(4.5*(temp-18)/14)
            xs=ruta[:paso+1]; ruta_line.set_data(xs,[0]*len(xs)); nodo.set_offsets([[temp,0]])
            etiqueta.set_text(f'Paso {paso+1}/{len(ruta)} · {temp} °C')
            return bodies+tails+eyes+[temp_text,state_text,mercury,ruta_line,nodo,etiqueta]

        anim=FuncAnimation(fig,actualizar,frames=frames,interval=180,blit=False,repeat=True)
        plt.close(fig); display(HTML(anim.to_jshtml()))

boton_simular.on_click(ejecutar_simulacion)
display(widgets.VBox([temperatura_control,boton_simular,salida]))
ejecutar_simulacion()

## Cómo explicar el grafo

1. **Nodo inicial:** temperatura simulada medida.
2. **Comparación:** el sistema verifica si está entre 24 °C y 26 °C.
3. **Decisión:** dentro del rango, mantiene el monitoreo; fuera, alerta y representa pasos correctivos.
4. **Ruta verde:** estados recorridos durante la demostración.
5. **Nodo final:** temperatura que vuelve al rango.

**Limitaciones:** lecturas ficticias, rango ilustrativo y corrección simulada. En una implementación real se requerirían sensores calibrados, validación por especie y mecanismos de seguridad. La lógica es de umbral, no IA predictiva ni heurística A*.